In [ ]:
from pathlib import Path
import pandas as pd

# 1. Loading

In [ ]:
folder = Path(r"C:\Users\z3553082\OneDrive - UNSW\Documents\CICCADA - Data\OEM analysis\OneDrive_1_6-18-2026\data_2025_01_12_alias_zipped")

files = list(folder.glob("*.parquet"))
df = pd.concat((pd.read_parquet(f) for f in files), ignore_index=True)

df.head()

In [ ]:
files

In [ ]:
alias_path = Path(r"C:\Users\z3553082\OneDrive - UNSW\Documents\CICCADA - Data\OEM analysis\OneDrive_1_6-18-2026\alias_mapping_alias_only.csv")

alias_df = pd.read_csv(alias_path)   # or read all parquet files if it's a folder
df = df.merge(
    alias_df[["alias", "zip_code"]],
    left_on="site_alias",
    right_on="alias",
    how="left"
)

df.head()

In [ ]:
df

# 2.

In [ ]:
site_id = "AUS908"

# 3phase:
site_id = "AUS631"

# single phase (raw sign fits)
site_id = "AUS515"

# single phase (flipped sign fits)
# site_id = "AUS352"

# single phase:
# site_id = "AUS1569"

test_site = df[df['site_alias'] == site_id]

In [ ]:
test_site

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# pick the day you want
selected_date = pd.Timestamp("2025-07-22") 

start = selected_date.normalize()
end = start + pd.Timedelta(days=1)

day_site = df[(df["site_alias"] == site_id) &
              (pd.to_datetime(df["timestamp"]) >= start) &
              (pd.to_datetime(df["timestamp"]) < end)].copy()

day_site["timestamp"] = pd.to_datetime(day_site["timestamp"])
day_site = day_site.sort_values("timestamp")

day_site["active_power_total"] = day_site[["active_power_1", "active_power_2", "active_power_3"]].sum(axis=1)
day_site["reactive_power_total"] = day_site[["reactive_power_1", "reactive_power_2", "reactive_power_3"]].sum(axis=1)
day_site["voltage_avg"] = day_site[["ac_voltage_1", "ac_voltage_2", "ac_voltage_3"]].mean(axis=1)

# Apparent power and power factor for phase 1
# S = sqrt(P^2 + Q^2)
day_site["apparent_power_1"] = (day_site["active_power_1"]**2 + day_site["reactive_power_1"]**2) ** 0.5
# PF = P / S
day_site["power_factor_1"] = day_site["active_power_1"] / day_site["apparent_power_1"]

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(14, 11))

day_site.plot(x="timestamp", y="active_power_total", ax=axes[0], legend=False, color="tab:blue")
axes[0].set_title(f"Active power for {site_id} on {start:%Y-%m-%d}")
axes[0].set_ylabel("Active power")
axes[0].grid(True, alpha=0.3)

day_site.plot(x="timestamp", y="reactive_power_total", ax=axes[1], legend=False, color="tab:orange")
axes[1].set_title("Reactive power")
axes[1].set_ylabel("Reactive power")
axes[1].grid(True, alpha=0.3)

day_site.plot(x="timestamp", y="voltage_avg", ax=axes[2], legend=False, color="tab:green")
axes[2].set_title("Average AC voltage")
axes[2].set_ylabel("Voltage")
axes[2].grid(True, alpha=0.3)

axes[2].xaxis.set_major_locator(mdates.HourLocator(interval=1))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.setp(axes[2].get_xticklabels(), rotation=45)

plt.tight_layout()

In [ ]:
day_site

In [ ]:
# Apparent power and power factor for phase 1
# S = sqrt(P^2 + Q^2)
day_site["apparent_power_1"] = (day_site["active_power_1"]**2 + day_site["reactive_power_1"]**2) ** 0.5

# PF = P / S
day_site["power_factor_1"] = day_site["active_power_1"] / day_site["apparent_power_1"]

In [ ]:
plot_df = day_site[["timestamp", "active_power_1", "reactive_power_1", "apparent_power_1", "power_factor_1"]].copy()
plot_df["timestamp"] = pd.to_datetime(plot_df["timestamp"])
plot_df = plot_df.sort_values("timestamp")

fig, axes = plt.subplots(4, 1, sharex=True, figsize=(14, 12))

for col, ax in zip(["active_power_1", "reactive_power_1", "apparent_power_1", "power_factor_1"], axes):
    plot_df.plot(x="timestamp", y=col, ax=ax, legend=False)
    ax.set_ylabel(col.replace("_", " ").title())
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Timestamp")
plt.tight_layout()